In [ ]:
# Notebook for watershed data  pre-processing for hydrological flood simulations
# Author: Omar Sallam
# Email: omarkhaledsallam@gmail.com

In [ ]:
#%% Library imports
from __future__ import annotations
import pygeohydro as gh
import pandas as pd
from py3dep import get_dem ,deg2mpm
import sys
import os
from shapely.geometry import box
from shapely.geometry import Point, MultiPoint
# Add river network inside HUC-8
from pynhd.network_tools import prepare_nhdplus
# === Standard Library Imports ===
from pathlib import Path
# === Scientific & Geospatial Libraries ===
import numpy as np
import matplotlib.pyplot as plt
import opt_einsum as oe
import rasterio
import shapely
import xarray as xr
from matplotlib.colors import LightSource
from scipy.spatial import KDTree
from shapely import ops
# === Datashader for Visualization ===
from datashader import transfer_functions as tf
from datashader import utils as ds_utils
from datashader.colors import Greys9, inferno
# === HyRiver Stack Libraries ===
import py3dep                             # For DEM access from 3DEP
import pygeoutils as geoutils            # Geometry and transformation tools
import pynhd        
from pygeohydro import WBD
from pynhd import WaterData
from pynhd import NLDI
from pyproj import Transformer
import mpld3
import rasterio.features
import geopandas as gpd
from shapely.ops import unary_union
from pygeohydro import NWIS  # not used here, but fine if you already have it
from pygeohydro.plot import cover_legends
import matplotlib.patches as mpatches
import dill
from pysheds.grid import Grid
import matplotlib.colors as colors
import geopandas as gpd
import shapely.geometry as geom
import rioxarray  # ensures .rio accessor is available
from scipy.optimize import curve_fit
import requests
from io import StringIO
import matplotlib.pyplot as plt
from requests_cache import field
from rasterio.features import rasterize


#%%
plt.rcParams.update({
    'axes.labelsize': 16,
    'font.size': 16,
    'mathtext.fontset': 'cm',
    'mathtext.default': 'it',
    'font.family': 'serif',
    'font.weight': 'normal'
})

In [ ]:
# Functions to be used 

def get_non_outlier_indices(df, column, k=1.5):
    """
    Returns indices of non-outliers in a column based on the IQR method.

    Parameters:
        df (pd.DataFrame): Input DataFrame
        column (str): Column name to check
        k (float): IQR multiplier (1.5 = standard, 3 = more tolerant)

    Returns:
        np.ndarray: indices of non-outlier rows
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)


    IQR = Q3 - Q1

    lower = Q1 - k * IQR
    upper = Q3 + k * IQR

    mask = (df[column] >= lower) & (df[column] <= upper)
    return df[mask].index.to_numpy()


def remove_zeros(df, column):
    """
    Removes rows where the given column has value = 0.

    Parameters:
        df (pd.DataFrame): Input DataFrame
        column (str): Column name to check

    Returns:
        df_nozero (pd.DataFrame): DataFrame with zero rows removed
        valid_idx (np.ndarray): Indices of rows kept
    """
    mask = df[column] != 0
    return df[mask].index.to_numpy()


# Power-law function: y = a * Q^b
def power_law(Q, a, b):
    return a * (Q ** b)



def min_dem_in_circle(dem,lat , lon, radius_m):
    """
    Extract the minimum DEM value within a circular buffer.

    Parameters
    ----------
    dem : xarray.DataArray
        DEM with a CRS (via rioxarray).
    lon : float
        Longitude of the point (degrees).
    lat : float
        Latitude of the point (degrees).
    radius_m : float
        Radius of the circle (meters).

    Returns
    -------
    float
        Minimum DEM value inside the circle (np.nan if no data).
    """

    # 1. Ensure DEM is in EPSG:5070
    dem_5070 = dem.rio.reproject("EPSG:5070")

    # 2. Make point GeoDataFrame in WGS84
    gdf = gpd.GeoDataFrame(
        geometry=[geom.Point(lon, lat)], 
        crs="EPSG:4326"
    )

    # 3. Reproject point to EPSG:5070
    gdf_5070 = gdf.to_crs("EPSG:5070")

    # 4. Make circular buffer in meters
    circle = gdf_5070.buffer(radius_m).iloc[0]

    # 5. Clip DEM to circle
    dem_clip = dem_5070.rio.clip([circle], dem_5070.rio.crs, drop=True)

    # 6. Return minimum value
    return float(dem_clip.min().values) 


In [ ]:
#%% Here we define the resolution and the Watershed ID 
resolution = 30
watershed_id = "07120004" # Desplaines # Example of HUC8
# watershed_id = "071402"  # Kaskaskia  # Example of HUC6
# watershed_id = "0714"   # Example of HUC4
# watershed_id = "07"   # Example of HUC2
huc_digit_number  = len(watershed_id)


In [ ]:
# Lets make paths for the data
import os
parent_path = 'watersheds/'+'huc' + str(huc_digit_number) + '_' + watershed_id + '/'
data_path = parent_path + rf'data_for_HiPIMS_{resolution}/'
os.makedirs(data_path, exist_ok=True)
figures_path = data_path + 'figures/'
os.makedirs(figures_path, exist_ok=True)

In [ ]:
#Get HUC# and its geometry
wbd = WBD(rf"huc{huc_digit_number}")
huc = wbd.byids(rf"huc{huc_digit_number}", [watershed_id])  # 
# Lets check the coordinate reference system (CRS) for the watershed polygon 
print(huc.crs)
m = huc.explore()
m.save(figures_path + "huc_watershed_boundary.html")
m


In [ ]:
# Lets get the waterlines 
#%%  Lets Get the river network
from pynhd import WaterData
wd = WaterData("nhdflowline_network")
# Get all water lines inside the HUC
flowlines = wd.bygeom(huc.geometry.iloc[0])
flowlines = prepare_nhdplus(flowlines, min_network_size=0, min_path_length=0, remove_isolated=True)
# Get the main flowline
main_flowline = pynhd.prepare_nhdplus(flowlines, 0, 0, 0, remove_isolated=True)
main_flowline = main_flowline[main_flowline.levelpathi == main_flowline.levelpathi.min()].to_crs("EPSG:4326").copy()


In [ ]:
# Let get the Gauges available in the watershed polygon and  their locations 
gagesii = WaterData("gagesii")
# Get all gages within the HUC-8 polygon
gages = gagesii.bygeom(huc.geometry.iloc[0])

# Unfortunately, these information do not include the altitude of the gage
# So we need to get it from NWIS
#%% Get the gage altitude from NWIS
# The info in gages frame down no include the vertical datum or description of the gage height
from pygeohydro import NWIS
nwis = NWIS()
for gage_idx , gage in gages.iterrows():
    site_id = gage['staid']  # replace with your gage ID
    queries = [{"site": [site_id]}]
    site_info = nwis.get_info(queries)
    gages.loc[gage_idx, "alt_va"] = site_info["alt_va"][0]

# Save the gage data file 
gages.to_file(data_path + "gages.geojson", driver="GeoJSON")



In [ ]:
# Plot the HUC-8 boundary and the river network and the main flow line and the gages
m = huc.boundary.explore(color="darkblue", name="HUC", tiles="OpenStreetMap")
# add flowlines
m = flowlines.explore(m=m, color="cyan", name="Flowlines")
m = main_flowline.explore(m=m, color="blue", name="Main Flowline")
m = gages.explore(m=m, marker_type="circle", color="red", name="Gages")

m.save(figures_path + "huc_watershed_waterlines.html")
m

In [ ]:
# Lets get the waterbody polygons for the watershed area 
from __future__ import annotations
import shapely
from pynhd import HP3D, NLDI

nhd3d = HP3D("waterbody")
waterbody = nhd3d.bygeom(huc.geometry.iloc[0])
waterbody = waterbody.set_geometry("geometry")

m = huc.boundary.explore(color="darkblue", name="HUC", tiles="OpenStreetMap")
waterbody.explore(m = m)
m.save(figures_path + "huc_watershed_waterbody.html")
m

In [ ]:
## Here we filter some waterbodies based on area

## Only keep the largest waterbody polygon area
waterbody_largest = waterbody.loc[[waterbody["shape_Area"].idxmax()]]

## Get waterbodies larger than certain area threshold
# waterbody_large = waterbody[waterbody["shape_Area"]>1000000]

## Combine the waterbody polygons
# waterbody_largest = unary_union(waterbody_largest)

# Lets plot the largest waterbody polygon
m = huc.boundary.explore(color="darkblue", name="HUC", tiles="OpenStreetMap")
waterbody_largest.explore(m = m)
# m = main_flowline.explore(m=m, color="blue", name="Main Flowline")
m = gages.explore(m=m, marker_type="circle", color="red", name="Gages")
m.save(figures_path + "huc_watershed_main_waterbody_gauges.html")
m

In [ ]:
## Lets download the DEM and save it 
dem = get_dem(huc.geometry.iloc[0], resolution=resolution)  # default is EPSG:5070 (projected meters)
projected_resolution = dem.rio.resolution()
print("DEM Resolution (x, y):", projected_resolution)
print(dem.shape)
print(dem.rio.crs)

dem_5070 = dem.rio.reproject("EPSG:5070",resolution=(resolution,resolution))  # reproject to lat/lon
projected_resolution = dem_5070.rio.resolution()
print("DEM Resolution (x, y):", projected_resolution)
print(dem_5070.shape)
print(dem_5070.rio.crs)


In [ ]:

# Save the DEM with different projections
dem_4326 = dem_5070.rio.reproject("EPSG:4326")  # reproject to lat/lon
dem_4326.rio.to_raster(
    data_path+"dem_4326.tif",
    compress="DEFLATE",
    tiled=True,
    predictor=2
)


dem_5070.rio.to_raster(
    data_path+"dem_5070.tif",
    compress="DEFLATE",
    tiled=True,
    predictor=2
)

# dem.rio.to_raster(
#     data_path+"dem.tif",
#     compress="DEFLATE",
#     tiled=True,
#     predictor=2
# )



In [ ]:
## Lets plot the DEM 
# Let pot for a fig
fig, ax = plt.subplots(figsize=(12, 10))

# Plot DEM
dem_4326.plot(
    ax=ax,
    cmap="terrain",
    alpha=1.0,
    zorder=1,
    cbar_kwargs={
        "label": "Elevation (m)",
        "shrink": 0.7,      # Shrink height (for vertical colorbars)
        "aspect": 30,       # Higher = thinner bar
        "pad": 0.02         # Space between colorbar and plot
    }
)
# Ensure vector CRS matches DEM
# huc8 = huc8.to_crs(dem.rio.crs)
# internal_10 = internal_10.to_crs(dem.rio.crs)

# Plot HUC-8 boundary
huc.boundary.plot(ax=ax, edgecolor="black", linewidth=2.5, label="HUC-8", zorder=4)
# Plot HUC-10 boundaries
huc.boundary.plot(ax=ax, edgecolor="black",ls='-' , alpha=0.4, linewidth=1.5, label="HUC-10", zorder=3)

i = 0
for _, gage in gages.iterrows():
    gage_geometry = gage.geometry
    point = list(gage.geometry.geoms)[0]
    ax.plot(point.x, point.y, 's', color="red", markersize=3, alpha = 0.8,zorder=5)
    # ax.text(point.x, point.y, f"{gage.staid}", fontsize=6, ha='left', va='bottom')
    ax.text(point.x, point.y, str(i), fontweight='bold',color ='red' ,fontsize=8, ha='left', va='bottom')
    i += 1


main_flowline.plot(ax=ax, linewidth=2, color="blue", label="Main river line", zorder=2)
flowlines.plot(ax=ax, linewidth=0.6, color="blue", label="River network", zorder=2)


# Final plot settings
# ax.set_title("DEM with HUC-8 and Internal HUC-10 Boundaries", fontsize=14)
# ax.set_title(" ", fontsize=1)
ax.set_title(None)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.legend(
    frameon=False,
    loc='lower center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=2,  # number of columns if many labels
    borderaxespad=0.
)# ax.set_axis_off()
plt.tight_layout()

plt.savefig(figures_path + "dem_gauges.png", dpi=300, bbox_inches="tight")


In [ ]:
# Lets improve the DEM using the Pysheds library
grid = Grid.from_raster(data_path+'dem_5070.tif')
dem_raster = grid.read_raster(data_path+'dem_5070.tif')

pits = grid.detect_pits(dem_raster)
dem_raster_pits_filled = grid.fill_pits(dem_raster)

flats = grid.detect_flats(dem_raster_pits_filled)
dem_raster_pits_filled_inflated = grid.resolve_flats(dem_raster_pits_filled)

In [ ]:
# Lets plot the flow accumumation for the original DEM and the modified DEM

# Compute flow direction based on corrected DEM
fdir = grid.flowdir(dem_raster)
# Compute flow accumulation based on computed flow direction
acc = grid.accumulation(fdir)
nan_indices = np.argwhere(np.isnan(dem_raster))
acc[nan_indices[:,0], nan_indices[:,1]] = -1
# "cubehelix"
plt.figure(figsize = (15,10), dpi = 200)
plt.subplot(1,3,1)
im = plt.imshow(acc, zorder=2,
               cmap='cubehelix',
               norm=colors.LogNorm(0.5, acc.max()/2),
               interpolation='bilinear'
               )
# plt.colorbar(im, label='Upstream Cells')
plt.title('No conditioning', size=14)
plt.xticks([])
plt.yticks([])

# Compute flow direction based on corrected DEM
fdir = grid.flowdir(dem_raster_pits_filled)
# Compute flow accumulation based on computed flow direction
acc = grid.accumulation(fdir)
nan_indices = np.argwhere(np.isnan(dem_raster))
acc[nan_indices[:,0], nan_indices[:,1]] = -1
# "cubehelix"
plt.subplot(1,3,2)
im = plt.imshow(acc, zorder=2,
               cmap='cubehelix',
               norm=colors.LogNorm(0.5, acc.max()/2),
               interpolation='bilinear'
               )
# plt.colorbar(im, label='Upstream Cells')
plt.title('Pits Filled', size=14)
plt.xticks([])
plt.yticks([])



# Compute flow direction based on corrected DEM
fdir = grid.flowdir(dem_raster_pits_filled_inflated)
# Compute flow accumulation based on computed flow direction
acc = grid.accumulation(fdir)
acc[nan_indices[:,0], nan_indices[:,1]] = -1

# fig, ax = plt.subplots(figsize=(8,6),dpi =200)
fig = plt.subplot(1,3,3)
im = plt.imshow(acc, zorder=2,
               cmap='cubehelix',
               norm=colors.LogNorm(0.5, acc.max()/2),
               interpolation='bilinear')
# plt.colorbar(im, label='Upstream Cells')
plt.title('Pits Filled + Flats Removed', size=14)
plt.xticks([])
plt.yticks([])
plt.tight_layout()




In [ ]:
# Lets show differences between the original DEM and the modified DEM using pysheds library

# dem = dem.rio.reproject("EPSG:5070")  # reproject to lat/lon


dem_pits_filled = dem_5070.copy()
dem_pits_filled.values = dem_raster_pits_filled

dem_pits_filled_inflated = dem_5070.copy()
dem_pits_filled_inflated.values = dem_raster_pits_filled_inflated



# print(dem.values)
plt.figure(figsize = (15,4))
plt.subplot(1,2,1)
plt.hist((dem_5070.values - dem_pits_filled.values).flatten(), bins=100, color='blue', log=True, alpha=0.4)
plt.title('DEM(Pits Filled)  -  DEM')
plt.xlabel('Elevation Difference (m)')
plt.subplot(1,2,2)
plt.hist((dem_pits_filled.values - dem_pits_filled_inflated.values).flatten(), bins=100, color='red', log=True, alpha=0.7)
plt.title('DEM(Pits Filled and Inflated)  -  DEM(Pits Filled)')
plt.xlabel('Elevation Difference (m)')
plt.tight_layout()
plt.savefig(figures_path + "pysheds_diffs_stats.png", dpi=300, bbox_inches="tight")


In [ ]:
# Lets Get the Elevations of DEM at locations of some USGS gauges 
# Main river indices 
gage_indices  = [0,19,20,25]
# Creeks Indices  
# gage_indices  = [23,28,29,14,17]

site_ids  =[]
alt_datums = []
elevation_values = []


for gage_idx in gage_indices:
    # print(f"Gage index: {gage_idx}, Gage ID: {gages.iloc[gage_idx]['staid']}")
    site_ids.append(gages.iloc[gage_idx]['staid'])
    alt_datums.append(gages.iloc[gage_idx]['alt_va'])
    elevation_value = min_dem_in_circle(dem, gages.iloc[gage_idx]['lat_gage'], gages.iloc[gage_idx]['lng_gage'], radius_m=100)
    elevation_values.append(elevation_value)



In [ ]:
# Plot the measurements and there curve fitting with the mean value for the dem correction value

start_date = "2010-09-24"
end_date = "2025-09-24"



fields  = ["chan_width" , "chan_depth" ,"chan_area", "chan_velocity" , "gage_height_va"  , "diff"]
ylabels = ["Channel Width (ft)" , "Channel Depth (ft)" ,"Channel Area (sq.ft)", "Channel Velocity (ft/s)" , "Gage Height (ft)" , "Gage Height - Depth (ft)" ]


burn_values = []
for field in fields:
    plt.figure(figsize=(40, 6))
    gage_counter = 0
    for  site_id in site_ids:

        url = (f"https://nwis.waterdata.usgs.gov/nwis/measurements?"
               f"site_no={site_id}&format=rdb_expanded"
               f"&begin_date={start_date}&end_date={end_date}")

        r = requests.get(url)
        df = pd.read_csv(StringIO(r.text), sep="\t", comment="#")

        #  'vert_vel_desc', 'chan_loc_cd'
        

        # Keep only rows that look like real dates
        mask = df["measurement_dt"].str.contains(r"\d{4}-\d{2}-\d{2}", na=False)
        df_clean = df[mask].copy()

        df_clean["measurement_dt"] = pd.to_datetime(df_clean["measurement_dt"], errors="coerce")
        df_clean["chan_discharge"] = pd.to_numeric(df_clean["chan_discharge"], errors="coerce")

       
        # print("Non-outlier indices for site", site_no, ":", non_outlier_indices)


        if field == "chan_depth":
            df_clean["chan_area"] = pd.to_numeric(df_clean["chan_area"], errors="coerce")
            df_clean["chan_width"] = pd.to_numeric(df_clean["chan_width"], errors="coerce")
            df_clean["chan_depth"] = df_clean["chan_area"] / df_clean["chan_width"]

        if field == "diff":
            df_clean["chan_area"] = pd.to_numeric(df_clean["chan_area"], errors="coerce")
            df_clean["chan_width"] = pd.to_numeric(df_clean["chan_width"], errors="coerce")
            df_clean["gage_height_va"] = pd.to_numeric(df_clean["gage_height_va"], errors="coerce")

            df_clean["chan_depth"] = df_clean["chan_area"] / df_clean["chan_width"]

            df_clean["diff"] = df_clean["gage_height_va"] - df_clean["chan_depth"] 



     

        plt.subplot(1, len(site_ids), gage_counter + 1)

        df_clean[field] = pd.to_numeric(df_clean[field], errors="coerce")

        # Drop rows where either x or y is NaN
        df_valid = df_clean.dropna(subset=["chan_discharge", field])

        non_outlier_indices = get_non_outlier_indices(df_valid, "chan_discharge", k=1.5)
        df_valid = df_valid.loc[non_outlier_indices]

        non_zeros_indices = remove_zeros(df_valid, "chan_discharge")
        df_valid = df_valid.loc[non_zeros_indices]

        df_valid = df_valid[df_valid["chan_discharge"] > 0]

        
        x = df_valid["chan_discharge"].values
        y = df_valid[field].values

        # Scatter plot
        plt.scatter(x, y, label="Measurements", alpha=0.7,marker='+', s=50, color='k')
        if field == "diff":

            mean_value  = y[y >= 0].mean()
            plt.axhline(mean_value, color='black', linestyle='--', label=rf'Mean +ve value = ${mean_value:.2f}$')
            burn_value = elevation_values[gage_counter]*3.28 - (mean_value+ alt_datums[gage_counter])
            # plt.title(rf'{alt_datums[gage_counter]}')
            # plt.title(rf'{burn_value}')

        # Curve fitting if enough valid points exist
        if field != "diff":
            if len(x) > 3:
                try:
                    popt, _ = curve_fit(power_law, x, y, maxfev=10000)
                    a, b = popt

                    # Smooth curve for plotting
                    x_fit = np.logspace(np.log10(min(x)), np.log10(max(x)), 200)
                    y_fit = power_law(x_fit, a, b)

                    plt.plot(x_fit, y_fit, 'r-', lw=2,
                            label=f"${{{a:.2f}}}  Q^{{{b:.2f}}}$")

                except Exception as e:
                    print(f"Curve fitting failed for site {site_id}: {e}")

            plt.xscale("log")
            plt.yscale("log")

        plt.xlabel("Discharge (cfs)")
        if gage_counter == 0:
            plt.ylabel(ylabels[fields.index(field)])
        
        plt.legend(frameon=False, loc='best')
        
        if field  == "diff":
            plt.title(f"[{gage_indices[gage_counter]}]: {site_id}, Depth correction value = {burn_value:.2f} ft")
            burn_values.append(burn_value)
        else: 
            plt.title(f"[{gage_indices[gage_counter]}]: {site_id}")

        plt.grid(True, which="both", ls="--", lw=0.5)
        
        plt.tight_layout()
        gage_counter += 1

burn_values = np.array(burn_values)
print(burn_values)
print('Depth Correction values: Mean = ', burn_values.mean(),  'Median = ' ,np.median(burn_values) ,'Min = ' ,burn_values.min(), 'Max = ' ,burn_values.max() )
plt.savefig(figures_path + "measurements_scatter_fit.png", dpi=300, bbox_inches="tight")


In [ ]:
# dem_pits_filled_inflated = dem_pits_filled_inflated.rio.reproject("EPSG:5070", resolution = (resolution,resolution))  # project to  5070
waterbody_largest = waterbody_largest.to_crs(dem_pits_filled_inflated.rio.crs)


In [ ]:
print(waterbody_largest.crs)
print(dem_pits_filled_inflated.rio.crs)

In [ ]:
# Lets Modify the DEM waterbody with the mean value
# Now we want to get the pixels in the DEM that intersects with the largest water body polygons 

transform = dem_pits_filled_inflated.rio.transform()
out_shape = dem_pits_filled_inflated.shape
mask_largest_water_body = rasterize(
    [(geom, 1) for geom in waterbody_largest.geometry],
    out_shape=out_shape,
    transform=transform,
    fill=0,
    dtype="uint8",
    all_touched=True # Too important 
)

from scipy.ndimage import binary_closing, binary_dilation
mask_largest_water_body = binary_closing(mask_largest_water_body, structure=np.ones((5,5))).astype(np.uint8)
# mask_largest_water_body = binary_dilation(mask_largest_water_body, iterations=1).astype(np.uint8)


# burn_depth = 1000                    # meters to lower, or replace with formula
burn_depth = burn_values.mean()*0.3048 # meters to lower, or replace with formula
dem_burned_5070 = dem_pits_filled_inflated.copy()
dem_burned_5070.values = np.where(mask_largest_water_body == 1, dem_pits_filled_inflated.values - burn_depth, dem_pits_filled_inflated.values)

print(dem_pits_filled_inflated.shape)
print(dem_burned_5070.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(dem_burned_5070.values, vert_exag=1, dx=1, dy=1)

plt.figure(figsize=(5,5), dpi=250)
# dem_burned.plot(cmap='terrain', add_colorbar=False)
plt.imshow(hillshade, cmap='gray', alpha=1, extent=(dem_burned_5070.x.min(), dem_burned_5070.x.max(), dem_burned_5070.y.min(), dem_burned_5070.y.max()))
plt.xticks([]); plt.yticks([])
plt.show()
plt.tight_layout()
plt.savefig(figures_path + "hillshade_dem.png", dpi=300, bbox_inches="tight")


In [ ]:
print(dem_burned_5070.shape)
print(mask_largest_water_body.shape)

In [ ]:
## Get the land cover data from the NLCD dataset
# Lets get land use 
nlcd = gh.nlcd_bygeom(huc, resolution=resolution, years={"cover": [2021]}, region="L48", crs="EPSG:4326")
cover_da = nlcd[huc.index[0]].cover_2021  # xarray.DataArray, categorical NLCD codes

# --- Ensure HUC-8 geometry is a single (multi)polygon in the same CRS ---
if huc.crs is None or huc.crs.to_string() != "EPSG:4326":
    huc = huc.to_crs("EPSG:4326")
huc_geom_unary_union = [unary_union(huc.geometry)]

# --- Build a raster mask: True inside polygon, False outside ---
mask_inside = rasterio.features.geometry_mask(
    huc_geom_unary_union,
    transform=cover_da.rio.transform(),
    invert=True,                  # keep inside polygon
    out_shape=cover_da.shape
)
# --- Apply mask so outside becomes NaN (will render as white) ---
land_cover= cover_da.where(mask_inside)

land_cover_4326 = land_cover.rio.reproject_match(dem_4326)


nlcd_names = {
    11: "Open Water",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land (Rock/Sand/Clay)",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}


In [ ]:
# Lets plot the land cover data

cmap, norm, levels = cover_legends()
cmap.set_bad(color="white")        # <- outside watershed will be white

# --- Figure / Axes ---
fig, ax = plt.subplots(figsize=(12, 10))

# Plot the NLCD (masked to HUC)
land_cover_4326.where(land_cover_4326 < 126).plot(
    ax=ax, cmap=cmap, norm=norm, levels=levels, add_colorbar=False
)

# --- Build legend only for classes present inside HUC ---
codes_present = sorted(
    int(c) for c in np.unique(land_cover_4326.values)
    if np.isfinite(c) and c < 126
)
patches = [
    mpatches.Patch(color=cmap(norm(code)), label=nlcd_names.get(code, str(code)))
    for code in codes_present
]
leg = ax.legend(
    handles=patches,
    bbox_to_anchor=(1.05, 1.0),
    loc="upper left",
    frameon=True,
    title="NLCD 2021 Classes"
)

# --- Plot HUC boundary on top ---
huc.boundary.plot(ax=ax, edgecolor="black", linewidth=2.0, zorder=3)

# --- Cosmetics ---
ax.set_title(None)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
# If you prefer no axes box/labels, uncomment:
# ax.set_axis_off()

plt.tight_layout()
plt.savefig(figures_path + "landuse.png", dpi=300, bbox_inches="tight")


In [ ]:
# Lets save the dem burned 
dem_burned_5070.rio.to_raster(
    data_path+"dem_burned_5070.tif",
    compress="DEFLATE",
    tiled=True,
    predictor=2
)


print(dem_burned_5070.rio.crs)
print(dem_burned_5070.shape)



# Add the match here :) 

# dem_burned_4326 = dem_burned_5070.rio.reproject("EPSG:4326")  # reproject to lat/lon
dem_burned_4326 = dem_burned_5070.rio.reproject_match(dem_4326)


dem_burned_4326.rio.to_raster(
    data_path+"dem_burned_4326.tif",
    compress="DEFLATE",
    tiled=True,
    predictor=2
)
print(dem_burned_4326.rio.crs)
print(dem_burned_4326.shape)



np.save(data_path + "mask_largest_water_body.npy", mask_largest_water_body)

print(mask_largest_water_body.shape)


In [ ]:

land_cover_4326.rio.to_raster(
    data_path + "landcover_4326.tif",
    compress="DEFLATE",
    tiled=True,
    predictor=2
)

land_cover_5070 = land_cover_4326.rio.reproject_match(dem_5070)
# land_cover = land_cover.rio.reproject("EPSG:5070",resolution = (resolution,resolution))  # or (geographic degrees)


print(land_cover_5070.rio.crs)
print(land_cover_5070.shape)



land_cover_5070.rio.to_raster(
    data_path + "landcover_5070.tif",
    compress="DEFLATE",
    tiled=True,
    predictor=2
)



In [ ]:
import dill
dill.dump_session(data_path + "watershed_data_preparation_session.pkl")

In [ ]:
import dill
# Restore workspace
dill.load_session(data_path + "watershed_data_preparation_session.pkl")
